In [2]:
import numpy as np
import os
from PIL import Image


image_folder = '/playpen-raid2/qinliu/data/ecDNA/images'
files = sorted(os.listdir(image_folder))
for file in files:
    file_path = os.path.join(image_folder, file)
    image_npy = np.array(Image.open(file_path))
    assert image_npy.shape == (2048, 2448), f'{file_path}: {image_npy.shape}'

In [26]:
subset_prefix = 'NCIH2170_Control_'
subset_train_file_path = '/playpen-raid2/qinliu/data/ecDNA/experiments_small/train.txt'
subset_val_file_path = '/playpen-raid2/qinliu/data/ecDNA/experiments_small/val.txt'

train_subset = set()
with open(subset_train_file_path) as train_f:
    for line in train_f:
        name = line.strip()
        train_subset.add(subset_prefix + name)

val_subset = set()
with open(subset_val_file_path) as val_f:
    for line in val_f:
        name = line.strip()
        val_subset.add(subset_prefix + name)

file_path = '/playpen-raid2/qinliu/data/ecDNA/datasets/records_0422_2024.txt'
full_set = set()
with open(file_path) as f:
    for line in f:
        name = line.strip()
        full_set.add(name)

print(len(full_set), len(train_subset), len(val_subset))
print(len(full_set - train_subset) == len(full_set) - len(train_subset))
print(len(full_set - val_subset) == len(full_set) - len(val_subset))


2990 100 28
True
True


In [27]:
import random
random.seed(0)

val_set = val_subset.union(
    random.sample(
        (full_set - train_subset) - val_subset, 
        600 - len(val_subset)
    )
)
train_set = full_set - val_set
assert len(val_set) == 600 
assert len(val_set) + len(train_set) == len(full_set)
assert len(val_set) == len(val_set.union(val_subset))
assert len(train_set) == len(train_set.union(train_subset))

In [28]:
train_set = sorted(train_set)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/train_0702_2024.txt', 'w') as f:
    for elm in train_set:
        f.write(elm + '\n')

In [29]:
val_set = sorted(val_set)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/val_0702_2024.txt', 'w') as f:
    for elm in val_set:
        f.write(elm + '\n')

### Dataset split for high-quality cases on validation set

In [5]:
file_path = '/playpen-raid2/qinliu/data/ecDNA/datasets/val_0422_2024.txt'
val_set = set()
with open(file_path) as f:
    for line in f:
        name = line.strip()
        val_set.add(name)
assert len(val_set) == 600

{'SUM159PT_DC_DMSO_72hr_33_Merge.png', 'NCIH2170_Antibiotics_DAPI_PS_73_Merge.png', 'SUM159PT_DC_JQ1_72hr_90_Merge.png', 'NCIH2170_Control_2_94.png', 'NCIH2170_Antibiotics_DAPI_PS_G_53_Merge.png', 'COLO320DM_qPCR_JQ1_24h_ctrl_4_Merge.png', 'NCIH2170_Antibiotics_DAPI_Blank_Control_12_Merge.png', 'NCIH2170_FACS_FISH_0723_Sorted_ctrl_52_Merge.png', 'SUM159PT_DC_JQ1_72hr_23_Merge.png', 'NCIH2170_Post_FACS_FISH_High_HER2_G3_57_check_Merge.png', 'SNU16_JC_JQ1_IC50_24h_52_Merge.png', 'SUM159PT_DC_JQ1_72hr_117_Merge.png', 'NCIH2170_Post_FACS_FISH_High_HER2_G3_67_check_Merge.png', 'NCIH2170_FACS_FISH_0723_Sorted_ctrl_95_Merge.png', 'NCIH2170_FACS_FISH_0223_High_HER2_13_Merge.png', 'SUM159PT_DC_JQ1_72hr_59_Merge.png', 'NCIH2170_JC_Lap_IC50_24h_14_Merge.png', 'NCIH2170_JC_Ner_IC50_72h_89_Merge.png', 'SNU16_JC_JQ1_IC50_24h_57_Merge.png', 'SUM159PT_DC_JQ1_72hr_108_Merge.png', 'SUM159PT_DC_JQ1_72hr_76_Merge.png', 'SUM159PT_DC_Control_55_Merge.png', 'NCIH2170_JC_JQ1_IC50_24h_12_Merge.png', 'NCIH2170_

In [8]:
import pandas as pd

quality_file = '/playpen-raid2/qinliu/data/ecDNA/datasets/quality_val_0422_2024.csv'
df = pd.read_csv(quality_file)

val_set_ncih_high = set()
for index, row in df.iterrows():
    score = row['score']
    name = row['name'].strip().split('.')[0] + '.png'
    if name.startswith('NCIH2170') and score >= 3.:
        val_set_ncih_high.add(name)

assert len(val_set_ncih_high) == 214

In [12]:
import random
random.seed(0)

full_set = val_set_ncih_high
val_set = set(random.sample(full_set, 50))
train_set = full_set - val_set
train_set_half = set(random.sample(train_set, len(train_set) // 2))

assert len(val_set) == 50
assert len(train_set) == 164
assert len(train_set_half) == 82
assert len(val_set) + len(train_set) == len(full_set)

In [13]:
train_set = sorted(train_set)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/train_0609_2024.txt', 'w') as f:
    for elm in train_set:
        f.write(elm + '\n')

train_set_half = sorted(train_set_half)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/train_half_0609_2024.txt', 'w') as f:
    for elm in train_set_half:
        f.write(elm + '\n')

val_set = sorted(val_set)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/val_0609_2024.txt', 'w') as f:
    for elm in val_set:
        f.write(elm + '\n')

## Dataset split for 1182 high-quality images

In [55]:
file_path = '/playpen-raid2/qinliu/data/ecDNA/datasets/records_0422_2024.txt'
full_set = set()
with open(file_path) as f:
    for line in f:
        name = line.strip()
        full_set.add(name)
print(len(full_set))        

2990


In [62]:
train_file_path = '/playpen-raid2/qinliu/data/ecDNA/datasets/train_0609_2024.txt'
train_set = set()
with open(train_file_path) as f:
    for line in f:
        name = line.strip()
        train_set.add(name)

val_file_path = '/playpen-raid2/qinliu/data/ecDNA/datasets/val_0609_2024.txt'
val_set = set()
with open(val_file_path) as f:
    for line in f:
        name = line.strip()
        val_set.add(name)

for elm in val_set:
    assert not elm in train_set, print(elm)
print(len(train_set), len(val_set))

164 50


In [63]:
import os

ROIs_folder = '/playpen-raid2/qinliu/projects/LabelEngine/data/ecDNA/ROIs'
ROIs = set()
files = sorted(os.listdir(ROIs_folder))
for file in files:
    ROIs.add(file)

print(val_set - ROIs)
val_set = val_set - (val_set - ROIs)

print(train_set - ROIs)
train_set = train_set - (train_set - ROIs)

set()
{'NCIH2170_Control_1_22.png', 'NCIH2170_Control_1_23.png', 'NCIH2170_FACS_FISH_0723_Sorted_ctrl_86_check_Merge.png'}


In [64]:
import random
random.seed(0)

val_set_0702 = val_set.union(
    random.sample(
        (ROIs - train_set) - val_set, 
        200 - len(val_set)
    )
)
train_set_0702 = ROIs - val_set_0702
print(len(val_set_0702), len(train_set_0702))

200 982


In [66]:
train_set_0702 = sorted(train_set_0702)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/train_0702_2024.txt', 'w') as f:
    for elm in train_set_0702:
        f.write(elm + '\n')

val_set_0702 = sorted(val_set_0702)
with open('/playpen-raid2/qinliu/data/ecDNA/datasets/val_0702_2024.txt', 'w') as f:
    for elm in val_set_0702:
        f.write(elm + '\n')        